<a href="https://colab.research.google.com/github/aashiqafathima823-sketch/amazonrating/blob/main/amazon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import files

uploaded = files.upload()

Saving amazon.csv to amazon (1).csv


In [15]:
import re
import numpy as np
import pandas as pd

# 1. Load dataset
df = pd.read_csv("amazon.csv")

# 2. Clean rating target (strip non-numeric artifacts like '|' and drop NaNs)
df["rating"] = pd.to_numeric(
    df["rating"].astype(str).str.replace("|", "", regex=False).str.strip(),
    errors="coerce",
)
df = df.dropna(subset=["rating"])

# 3. Clean and merge product information and customer reviews
text_cols = ["product_name", "category", "review_title", "review_content"]
df["text"] = (
    df[text_cols]
    .fillna("")
    .agg(" ".join, axis=1)
    .apply(lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x.lower()).strip())
)

print(f"Data shape after cleaning: {df.shape}")
print(df[["text", "rating"]].head(2))

Data shape after cleaning: (1464, 17)
                                                text  rating
0  wayona nylon braided usb to lightning fast cha...     4.2
1  ambrane unbreakable 60w  3a fast charging 15m ...     4.0


In [16]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

VOCAB_SIZE = 5000
MAX_LEN = 100

# 1. Fit tokenizer on processed review text
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

# 2. Convert to padded sequences
X = pad_sequences(
    tokenizer.texts_to_sequences(df["text"]), maxlen=MAX_LEN, padding="post"
)
y = df["rating"].values.astype(np.float32)

# 3. Normalize ratings to [0, 1] range for stable gradient descent
y_norm = (y - 1.0) / 4.0

# 4. Train/Test Split (80/20)
X_train, X_test, y_train, y_test, y_train_norm, y_test_norm = train_test_split(
    X, y, y_norm, test_size=0.2, random_state=42
)

print(f"Train samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

Train samples: 1171 | Test samples: 293


In [25]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.models import Sequential

# 1. Custom Keras Accuracy Metric: within +/- 0.75 stars
def rating_accuracy(y_true, y_pred):
    tolerance_norm = 0.75 / 4.0
    correct = tf.less_equal(tf.abs(y_true - y_pred), tolerance_norm)
    return tf.reduce_mean(tf.cast(correct, tf.float32)) * 100.0

# 2. Build cuDNN-compatible Bidirectional LSTM Architecture
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=64),
    SpatialDropout1D(0.2),
    # Notice: No recurrent_dropout, allowing fast cuDNN GPU acceleration
    Bidirectional(LSTM(64, dropout=0.2)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.Huber(delta=0.1),
    metrics=["mae", rating_accuracy]
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_3             │ (None, 100, 64)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,209 (1.49 MB)

 Trainable params: 390,209 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [26]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

# Train the model
history = model.fit(
    X_train,
    y_train_norm,
    validation_data=(X_test, y_test_norm),
    epochs=25,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

# Evaluate model
test_loss, test_mae, test_acc = model.evaluate(X_test, y_test_norm, verbose=0)
print("\n" + "=" * 45)
print(f"EVALUATION ACCURACY:        {test_acc:.2f}%")
print(f"Mean Absolute Error (Stars): {test_mae * 4.0:.4f}")
print("=" * 45)

Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - loss: 0.0078 - mae: 0.1181 - rating_accuracy: 76.8370 - val_loss: 0.0027 - val_mae: 0.0587 - val_rating_accuracy: 94.6797 - learning_rate: 0.0010
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0028 - mae: 0.0608 - rating_accuracy: 96.8838 - val_loss: 0.0021 - val_mae: 0.0511 - val_rating_accuracy: 96.2617 - learning_rate: 0.0010
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0022 - mae: 0.0528 - rating_accuracy: 97.2049 - val_loss: 0.0020 - val_mae: 0.0494 - val_rating_accuracy: 95.7734 - learning_rate: 0.0010
Epoch 4/25
36/37 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0021 - mae: 0.0506 - rating_accuracy: 96.9412
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0020 - mae: 0.0493 - rating_accuracy: 96.6553 - val_loss: 0.0020 - val_mae: 0.0486 - val_rating_accuracy: 96.2223 - learning_rate: 0.0010
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━

In [27]:
# 1. Compare actual vs predicted on test set
preds_norm = model.predict(X_test).flatten()
preds = preds_norm * 4.0 + 1.0  # Rescale back to 1.0 - 5.0 scale

comparison = pd.DataFrame({
    "Actual Rating": y_test[:10],
    "Predicted Rating": np.round(preds[:10], 2),
    "Absolute Error": np.round(np.abs(y_test[:10] - preds[:10]), 2),
})

print("--- Comparison (First 10 Test Samples) ---")
print(comparison.to_string(index=False))

# 2. Demonstrate predictions on completely unseen reviews
unseen_reviews = [
    "Amazing product, high quality build, and worked as expected!",
    "Terrible build quality, stopped working after just one day.",
    "Decent quality for the price, nothing extraordinary.",
]

unseen_seqs = pad_sequences(
    tokenizer.texts_to_sequences(unseen_reviews),
    maxlen=MAX_LEN,
    padding="post",
)
unseen_preds = model.predict(unseen_seqs).flatten() * 4.0 + 1.0

print("\n--- Predictions on Unseen Reviews ---")
for rev, score in zip(unseen_reviews, unseen_preds):
  print(f'Review: "{rev}"')
  print(f"Predicted Rating: {score:.2f} / 5.0\n")


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step
--- Comparison (First 10 Test Samples) ---
 Actual Rating  Predicted Rating  Absolute Error
           4.2              4.04            0.16
           3.7              4.01            0.31
           3.3              3.63            0.33
           4.3              4.07            0.23
           3.9              4.25            0.35
           4.5              4.15            0.35
           4.1              4.15            0.05
           4.1              4.06            0.04
           4.2              4.17            0.03
           3.6              4.15            0.55
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step

--- Predictions on Unseen Reviews ---
Review: "Amazing product, high quality build, and worked as expected!"
Predicted Rating: 3.88 / 5.0

Review: "Terrible build quality, stopped working after just one day."
Predicted Rating: 3.74 / 5.0

Review: "Decent quality for the price, nothing extraordinary."
Predicted Rating: 3.82 / 5.0



In [28]:
# Sample test cases spanning different sentiments and product categories
test_samples = [
    {
        "type": "5-Star (Strongly Positive)",
        "input": (
            "OnePlus Nord Buds 2 Wireless Earbuds Electronics Audio Must Buy!"
            " Absolutely mindblowing sound quality, premium bass, and superb"
            " battery life. Worth every penny!"
        ),
    },
    {
        "type": "4-Star (Positive with minor critique)",
        "input": (
            "Amazon Basics Ergonomic Office Chair Home Furniture Good Chair"
            " Comfortable back support and sturdy frame. Setup took around 30"
            " minutes, overall quite satisfied."
        ),
    },
    {
        "type": "3-Star (Average / Neutral)",
        "input": (
            "Milton Stainless Steel Water Bottle Kitchen Dining Average Product"
            " Keeps water cold for roughly 4 hours. Build quality is okay, but"
            " the cap feels a bit flimsy."
        ),
    },
    {
        "type": "2-Star (Below Average)",
        "input": (
            "Portronics USB-C Fast Charger Mobile Accessories Not Fast Enough"
            " Charges slower than the original cable and gets uncomfortably hot"
            " within twenty minutes of use."
        ),
    },
    {
        "type": "1-Star (Extremely Negative)",
        "input": (
            "Zebronics Wired Gaming Mouse Computers Accessories Total Waste of"
            " Money Stopped working on the second day. Buttons are completely"
            " unresponsive and plastic feels cheap."
        ),
    },
]

# Preprocess & encode the sample inputs
sample_texts = [s["input"] for s in test_samples]
sample_seqs = pad_sequences(
    tokenizer.texts_to_sequences(sample_texts), maxlen=MAX_LEN, padding="post"
)

# Predict & scale back to 1.0 - 5.0 range
sample_preds = model.predict(sample_seqs).flatten() * 4.0 + 1.0
sample_preds = np.clip(sample_preds, 1.0, 5.0)

# Display results
print("=" * 70)
print(f"{'Category / Intent':<35} | {'Predicted Rating'}")
print("=" * 70)
for sample, pred in zip(test_samples, sample_preds):
  print(f"{sample['type']:<35} | {pred:.2f} / 5.0")
print("=" * 70)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Category / Intent                   | Predicted Rating
5-Star (Strongly Positive)          | 4.05 / 5.0
4-Star (Positive with minor critique) | 4.18 / 5.0
3-Star (Average / Neutral)          | 3.99 / 5.0
2-Star (Below Average)              | 4.11 / 5.0
1-Star (Extremely Negative)         | 3.89 / 5.0
